# Notebook 03: Evaluación y Análisis de Errores


---
### Objetivo
Evaluar el modelo sobre el conjunto de validación propio (40 preguntas distribuidas en 15 fáciles, 15 medias y 10 difíciles) en las bases sistema_academico y clinica_salud.
Calcula la métrica oficial de Exactitud por Ejecución (EX) ponderada por nivel (pesos 1, 2 y 3) y genera el análisis detallado de errores por categoría.

In [1]:
# Setup universal
import os, sys, subprocess
from pathlib import Path

# Detectar si estamos en Google Colab
EN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if EN_COLAB:
    import getpass
    REPO = "Albu231311/Deep-Proyecto3"
    NOMBRE = "Albu231311"
    CORREO = "alb231311@uvg.edu.gt"
    if not os.path.exists("/content/repo"):
        token = getpass.getpass("GitHub token: ")
        subprocess.run(f"git clone https://{token}@github.com/{REPO}.git /content/repo", shell=True, check=True)
    os.chdir("/content/repo")
    subprocess.run("git pull", shell=True, check=True)
    subprocess.run(f'git config user.name "{NOMBRE}"', shell=True)
    subprocess.run(f'git config user.email "{CORREO}"', shell=True)
    subprocess.run("pip install -q -r requirements.txt", shell=True, check=True)
    ROOT_DIR = Path("/content/repo")
else:
    
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
    ROOT_DIR = Path.cwd().resolve()

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print(f" Entorno listo. Directorio de trabajo: {os.getcwd()} | En Colab: {EN_COLAB}")


 Entorno listo. Directorio de trabajo: C:\Users\eagi5\Desktop\UVG\uvg-deep-learning\Proyecto#3\Deep-Proyecto3 | En Colab: False


## 1. Carga de Esquemas y Preguntas de Validación

Cargamos los 2 esquemas en MySQL y leemos datos/validacion/val.jsonl.

In [2]:
import json
from pathlib import Path
from src.mysql_utils import cargar_esquema, ejecutar

esq_dir = Path("datos/validacion/esquemas_val")
if not esq_dir.exists() and Path("../datos/validacion/esquemas_val").exists():
    esq_dir = Path("../datos/validacion/esquemas_val")

cargar_esquema(str(esq_dir / "sistema_academico.sql"), "sistema_academico")
cargar_esquema(str(esq_dir / "clinica_salud.sql"), "clinica_salud")

val_file = Path("datos/validacion/val.jsonl")
if not val_file.exists() and Path("../datos/validacion/val.jsonl").exists():
    val_file = Path("../datos/validacion/val.jsonl")

with open(val_file, "r", encoding="utf-8") as f:
    val_preguntas = [json.loads(line) for line in f]

print(f"Total de preguntas de validación: {len(val_preguntas)}")


Total de preguntas de validación: 40


## 2. Evaluación por Ejecución y Cálculo de EX Ponderado

Evaluamos la referencia con src/evaluador.py para certificar la métrica oficial.

In [3]:
from src.evaluador import evaluar_validacion

def predictor_referencia(esquema, pregunta, ejecutar_fn):
    for p in val_preguntas:
        if p["pregunta"] == pregunta:
            return p["sql_referencia"]
    return "SELECT 1"

db_map = {
    "sistema_academico": "sistema_academico",
    "clinica_salud": "clinica_salud"
}

resultado_eval = evaluar_validacion(val_preguntas, predictor_referencia, db_map)
print(f"EX Ponderado Global : {resultado_eval['ex_ponderado'] * 100:.2f}%")
print(f"EX por Nivel        : {resultado_eval['ex_por_nivel']}")
print(f"Conteo de Errores   : {resultado_eval['conteo_errores']}")


EX Ponderado Global : 100.00%
EX por Nivel        : {'facil': 1.0, 'media': 1.0, 'dificil': 1.0}
Conteo de Errores   : {}


## 3. Exportación de Resultados

Guardamos la tabla detallada en resultados/validacion_propia.csv.

In [4]:
from src.evaluador import guardar_csv

out_csv = Path("resultados/validacion_propia.csv")
if not out_csv.parent.exists() and Path("../resultados").exists():
    out_csv = Path("../resultados/validacion_propia.csv")

guardar_csv(resultado_eval["resultados"], str(out_csv), version="baseline_referencia")
print(f"Resultados guardados en: {out_csv}")


Resultados guardados en: resultados\validacion_propia.csv


## 4. Evaluación de los modelos

Se evalúa cada configuración con **exactamente** la función de entrega (`solucion.generar_sql`) y el mismo prompt:

| Versión | Descripción |
|---|---|
| `baseline` | Modelo base sin adapter (`usar_adapter: false`) |
| `lora_r8` | Config 1: LoRA r=8, alpha=16 — dataset v1 (preguntas de Spider tal cual) |
| `lora_r32` | Config 2: LoRA r=32, alpha=64 — dataset v1 |
| `lora_r8_v2`, `lora_r32_v2` | Mismas configs con dataset v2 (preguntas con indicación de columnas, notebook 02b) |
| `..._ep1` | Adapter v2 al final de la época 1 (de 2) |
| `<mejor>_autocorreccion` | Experimento de inferencia: la mejor config + corrección con `ejecutar` (máx. 3 llamadas) |

Para cada pregunta se registra también el tiempo de generación (límite: 30 s) y cuántas veces se llamó a `ejecutar`.

In [ ]:
import os, time
os.environ["SOLUCION_NO_CARGAR"] = "1"   # cargamos cada configuración a mano
import solucion
from src.evaluador import evaluar_validacion
from src.mysql_utils import ejecutar

CONFIGS = {"baseline": dict(usar_adapter=False)}
# v1: preguntas de Spider tal cual | v2: preguntas con indicación de columnas (notebook 02b)
# _ep1: adapter al final de la primera época (de 2)
for nombre in ["r8", "r32", "r8_v2_ep1", "r8_v2", "r32_v2_ep1", "r32_v2"]:
    CONFIGS[f"lora_{nombre}"] = dict(usar_adapter=True, adapter=f"adapters_experimentos/adapter_{nombre}")
DIR_EVAL = Path("resultados/evaluaciones")
DIR_EVAL.mkdir(parents=True, exist_ok=True)
RESULTADOS = {}


def cargar(nombre, **kw):
    t = time.time()
    solucion.cargar_modelo(**kw)
    seg = time.time() - t
    print(f"[{nombre}] modelo cargado en {seg:.1f} s (límite: 300 s)")
    return seg


def evaluar_config(nombre, autocorreccion=False):
    # Evalúa la configuración ya cargada en solucion.py y guarda el resultado en JSON
    tiempos, llamadas = [], []

    def gen(esquema, pregunta, ej):
        n = [0]
        def ej_contado(sql):
            n[0] += 1
            return ej(sql)
        t = time.time()
        try:
            return solucion.generar_sql(esquema, pregunta, ej_contado if autocorreccion else None)
        finally:
            tiempos.append(round(time.time() - t, 2))
            llamadas.append(n[0])

    res = evaluar_validacion(val_preguntas, gen, db_map)
    for p, r, t, n in zip(val_preguntas, res["resultados"], tiempos, llamadas):
        # ejecuta = la consulta corre sin error en MySQL (aunque el resultado sea incorrecto)
        ejecuta = ejecutar(r["sql_pred"], db_map[p["esquema_id"]])["ok"]
        r.update(version=nombre, segundos=t, llamadas_ejecutar=n, ejecuta=ejecuta)
    res["tiempo_max_s"] = max(tiempos)
    res["tiempo_medio_s"] = sum(tiempos) / len(tiempos)
    with open(DIR_EVAL / f"{nombre}.json", "w", encoding="utf-8") as f:
        json.dump(res, f, indent=2, ensure_ascii=False, default=str)
    print(f"[{nombre}] {res['resumen']}")
    print(f"[{nombre}] tiempo por pregunta: medio {res['tiempo_medio_s']:.1f} s, máx {res['tiempo_max_s']:.1f} s")
    return res

### 4.1 Determinismo

La rúbrica penaliza salidas no deterministas (−10). Se genera dos veces la respuesta a las mismas preguntas con el modelo base y se verifica que sean idénticas.

In [ ]:
from src.mysql_utils import obtener_esquema

cargar("baseline", usar_adapter=False)
muestra = val_preguntas[::8]
for p in muestra:
    esquema = obtener_esquema(db_map[p["esquema_id"]])
    a = solucion.generar_sql(esquema, p["pregunta"])
    b = solucion.generar_sql(esquema, p["pregunta"])
    assert a == b, f"No determinista en {p['id']}:\n{a}\n{b}"
print(f"Determinismo OK en {len(muestra)} preguntas")

### 4.2 Baseline, Config 1 y Config 2

Si `REUSAR = True`, las configuraciones ya evaluadas se leen de `resultados/evaluaciones/` en vez de volver a generar.

In [ ]:
REUSAR = True

for nombre, kw in CONFIGS.items():
    archivo = DIR_EVAL / f"{nombre}.json"
    if REUSAR and archivo.exists():
        RESULTADOS[nombre] = json.load(open(archivo, encoding="utf-8"))
        print(f"[{nombre}] leído de {archivo}: EX = {RESULTADOS[nombre]['ex_ponderado']:.4f}")
        continue
    if kw.get("adapter") and not Path(kw["adapter"]).exists():
        print(f"[{nombre}] se omite: no existe {kw['adapter']} (¿ya terminó el entrenamiento?)")
        continue
    seg = cargar(nombre, **kw)
    RESULTADOS[nombre] = evaluar_config(nombre)
    RESULTADOS[nombre]["carga_s"] = seg

## 5. Experimento de inferencia: autocorrección con `ejecutar`

**Hipótesis:** cuando la consulta generada falla en MySQL (columna inexistente, `ONLY_FULL_GROUP_BY`, sintaxis), mostrarle al modelo el mensaje de error le permite corregirla. Esperamos que reduzca los errores de ejecución y suba el EX, sin ayudar en los casos donde la consulta ejecuta pero devuelve un resultado distinto (ahí `ejecutar` devuelve `ok=True` y no hay señal de error).

Se aplica a la mejor configuración de LoRA según EX.

In [ ]:
candidatas = {n: r for n, r in RESULTADOS.items()
              if n.startswith("lora_") and not n.endswith("_autocorreccion")}
MEJOR, nombre_auto = None, None
if not candidatas:
    print("Todavía no hay configuraciones LoRA evaluadas; se omite el experimento.")
else:
    MEJOR = max(candidatas, key=lambda n: candidatas[n]["ex_ponderado"])
    print("Mejor configuración:", MEJOR, f"(EX = {candidatas[MEJOR]['ex_ponderado']:.4f})")

    nombre_auto = f"{MEJOR}_autocorreccion"
    archivo = DIR_EVAL / f"{nombre_auto}.json"
    if REUSAR and archivo.exists():
        RESULTADOS[nombre_auto] = json.load(open(archivo, encoding="utf-8"))
        print(f"[{nombre_auto}] leído: EX = {RESULTADOS[nombre_auto]['ex_ponderado']:.4f}")
    else:
        cargar(MEJOR, **CONFIGS[MEJOR])
        RESULTADOS[nombre_auto] = evaluar_config(nombre_auto, autocorreccion=True)

## 6. Resumen comparativo

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def errores_ejecucion(res):
    # Consultas que MySQL rechaza (no confundir con "columna_inexistente" del evaluador,
    # que también incluye consultas que ejecutan pero no devuelven las columnas pedidas)
    return sum(1 for r in res["resultados"] if not r.get("ejecuta", True))

resumen = pd.DataFrame([{
    "version": n,
    "EX": round(r["ex_ponderado"], 4),
    "EX_facil": round(r["ex_por_nivel"]["facil"], 4),
    "EX_media": round(r["ex_por_nivel"]["media"], 4),
    "EX_dificil": round(r["ex_por_nivel"]["dificil"], 4),
    "aciertos": f"{sum(x['acierto'] for x in r['resultados'])}/{len(r['resultados'])}",
    "errores_de_ejecucion": errores_ejecucion(r),
    "tiempo_medio_s": round(r.get("tiempo_medio_s", 0), 2),
    "tiempo_max_s": r.get("tiempo_max_s"),
} for n, r in RESULTADOS.items()])

if "baseline" in RESULTADOS:
    resumen["mejora_vs_baseline"] = (resumen["EX"] - RESULTADOS["baseline"]["ex_ponderado"]).round(4)
resumen.to_csv("resultados/resumen_ex.csv", index=False)

ax = resumen.plot.bar(x="version", y=["EX_facil", "EX_media", "EX_dificil", "EX"], figsize=(9, 4.5), rot=0)
ax.set_ylim(0, 1); ax.set_ylabel("EX"); ax.set_title("EX ponderado por configuración y nivel")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig("resultados/ex_por_configuracion.png", dpi=150); plt.show()
resumen

## 7. Análisis de errores por categoría

Categorías del evaluador: sintaxis, tabla inexistente, columna inexistente, JOIN, agregación, filtro y resultado distinto.

In [ ]:
filas = [r for res in RESULTADOS.values() for r in res["resultados"]]
df = pd.DataFrame(filas)

errores = df[~df["acierto"]]
tabla = pd.crosstab(errores["version"], errores["categoria_error"], margins=True, margins_name="total")
tabla.to_csv("resultados/errores_por_categoria.csv")
tabla

In [ ]:
# Errores por nivel de dificultad
pd.crosstab([errores["version"], errores["nivel"]], errores["categoria_error"])

In [ ]:
# Ejemplos concretos de errores de la mejor configuración (o del baseline si aún no hay LoRA)
VER = MEJOR or "baseline"
print(f"Errores de {VER}:")
print()
for _, r in df[(df["version"] == VER) & (~df["acierto"])].iterrows():
    print(f"[{r['id']} · {r['nivel']} · {r['categoria_error']}] {r['pregunta']}")
    print(f"   pred: {r['sql_pred']}")
    print(f"   ref : {r['sql_ref']}\n")

In [ ]:
# Efecto de la autocorrección pregunta por pregunta
if nombre_auto in RESULTADOS:
    a = df[df["version"] == MEJOR].set_index("id")
    b = df[df["version"] == nombre_auto].set_index("id")
    arregladas = a.index[(~a["acierto"]) & (b["acierto"])]
    rotas = a.index[(a["acierto"]) & (~b["acierto"])]
    print(f"Preguntas arregladas por la autocorrección: {list(arregladas)}")
    print(f"Preguntas que empeoraron: {list(rotas)}")
    print(f"Llamadas a ejecutar: {b['llamadas_ejecutar'].value_counts().sort_index().to_dict()}")

## 8. Exportación de resultados

`resultados/validacion_propia.csv` queda con una fila por pregunta y versión (incluida la verificación de la referencia de la sección 2).

In [ ]:
cols = ["id", "nivel", "pregunta", "sql_pred", "sql_ref", "acierto", "categoria_error",
        "version", "ejecuta", "segundos", "llamadas_ejecutar"]
referencia = pd.DataFrame(resultado_eval["resultados"]).assign(version="referencia")
salida = pd.concat([referencia, df], ignore_index=True).reindex(columns=cols)
salida.to_csv(out_csv, index=False, encoding="utf-8")
print(f"{len(salida)} filas guardadas en {out_csv}")
salida.groupby("version")["acierto"].mean()